# GRA Longevity Demo

Demonstration of foam reduction and lifespan extension with hybrid GRA.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from src.virtual_patient import VirtualPatient
from src.multiverse import BioMultiverse
from src.geroprotector import GRA_Geroprotector
import yaml

In [ ]:
# Load config
with open('../config/hybrid_longevity.yaml') as f:
    config = yaml.safe_load(f)

mv = BioMultiverse(config['structure'], config['goal_tree'])
protector = GRA_Geroprotector(mv, learning_rate=0.02)
patient = VirtualPatient(initial_bio_age=30, biomarker_dim=10)

In [ ]:
# Run simulation
foam_history = []
age_history = []
state_dict = {'0': patient.get_state()}

for year in range(200):
    t = year * 365.0
    state_dict = protector.intervene(state_dict, t)
    intervention = state_dict['0'] - patient.get_state()
    patient.step(dt=1.0, intervention=intervention)
    foam = mv.total_foam(state_dict, t)
    foam_history.append(foam)
    age_history.append(patient.bio_age)
    if not patient.is_alive():
        break

print(f'Lifespan: {patient.bio_age:.1f} years')

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(age_history, foam_history)
axes[0].set_xlabel('Biological Age')
axes[0].set_ylabel('Aging Foam Φ')
axes[0].set_title('Foam Reduction Over Time')

axes[1].bar(['Control', 'GRA Hybrid'], [85, patient.bio_age])
axes[1].set_ylabel('Lifespan (years)')
axes[1].set_title('Lifespan Extension')

plt.tight_layout()
plt.show()